In [1]:
# IMPORT LIBRARIES

import os
import random
import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import models, transforms

from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

import kagglehub

print("Libraries loaded successfully.")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Libraries loaded successfully.
PyTorch version: 2.11.0+cu130
CUDA available: True


In [2]:
# UPLOAD THE SAME MRI-TRAINED EFFICIENTNET-B0 CHECKPOINT
# USED IN THE HQNN EXPERIMENT

from google.colab import files

uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError(
        "Please upload exactly one EfficientNet-B0 .pth checkpoint."
    )

uploaded_filename = next(iter(uploaded))

if not uploaded_filename.lower().endswith(".pth"):
    raise ValueError(
        f"Expected a .pth checkpoint, received: {uploaded_filename}"
    )

EFFICIENTNET_CHECKPOINT = f"/content/{uploaded_filename}"

if not os.path.exists(EFFICIENTNET_CHECKPOINT):
    raise FileNotFoundError(
        f"Checkpoint not found: {EFFICIENTNET_CHECKPOINT}"
    )

print("Checkpoint uploaded successfully.")
print("Filename:", uploaded_filename)
print("Path:", EFFICIENTNET_CHECKPOINT)
print(
    "Size:",
    round(
        os.path.getsize(EFFICIENTNET_CHECKPOINT)
        / (1024 ** 2),
        2
    ),
    "MB"
)

Saving best_efficientnet_b0_finetuned.pth to best_efficientnet_b0_finetuned.pth
Checkpoint uploaded successfully.
Filename: best_efficientnet_b0_finetuned.pth
Path: /content/best_efficientnet_b0_finetuned.pth
Size: 15.6 MB


In [3]:
# REPRODUCIBILITY SETTINGS

SEED = 42


def set_global_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


set_global_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Random seed fixed:", SEED)

Random seed fixed: 42


In [4]:
# DOWNLOAD DATASET AND CREATE SAME 70/15/15 SPLIT

path = kagglehub.dataset_download(
    "masoudnickparvar/brain-tumor-mri-dataset"
)

classes = [
    "glioma",
    "meningioma",
    "notumor",
    "pituitary"
]

all_files = []
all_labels = []


for class_name in classes:

    for folder in ["Training", "Testing"]:

        class_path = (
            Path(path)
            / folder
            / class_name
        )

        for file_path in class_path.iterdir():

            if file_path.is_file():

                all_files.append(
                    str(file_path)
                )

                all_labels.append(
                    class_name
                )


# 70% TRAIN
# 30% TEMP

train_files, temp_files, train_labels, temp_labels = (
    train_test_split(
        all_files,
        all_labels,
        test_size=0.30,
        random_state=42,
        stratify=all_labels
    )
)


# TEMP → 15% VALIDATION / 15% TEST

val_files, test_files, val_labels, test_labels = (
    train_test_split(
        temp_files,
        temp_labels,
        test_size=0.50,
        random_state=42,
        stratify=temp_labels
    )
)


print("Dataset path:", path)
print("Total images:", len(all_files))

print(
    "Overall distribution:",
    Counter(all_labels)
)

print(
    "Training:",
    len(train_files),
    Counter(train_labels)
)

print(
    "Validation:",
    len(val_files),
    Counter(val_labels)
)

print(
    "Testing:",
    len(test_files),
    Counter(test_labels)
)

Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.
Dataset path: /kaggle/input/brain-tumor-mri-dataset
Total images: 7200
Overall distribution: Counter({'glioma': 1800, 'meningioma': 1800, 'notumor': 1800, 'pituitary': 1800})
Training: 5040 Counter({'notumor': 1260, 'pituitary': 1260, 'glioma': 1260, 'meningioma': 1260})
Validation: 1080 Counter({'notumor': 270, 'glioma': 270, 'meningioma': 270, 'pituitary': 270})
Testing: 1080 Counter({'meningioma': 270, 'notumor': 270, 'glioma': 270, 'pituitary': 270})


In [6]:
# IMAGE PREPROCESSING AND DATALOADERS

class_to_idx = {
    "glioma": 0,
    "meningioma": 1,
    "notumor": 2,
    "pituitary": 3
}


train_transform = transforms.Compose([

    transforms.Resize(
        (224, 224)
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


val_test_transform = transforms.Compose([

    transforms.Resize(
        (224, 224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


class BrainTumorDataset(Dataset):

    def __init__(
        self,
        files,
        labels,
        transform=None
    ):

        self.files = files
        self.labels = labels
        self.transform = transform


    def __len__(self):

        return len(self.files)


    def __getitem__(self, idx):

        image = Image.open(
            self.files[idx]
        ).convert("RGB")

        if self.transform:

            image = self.transform(
                image
            )

        label = class_to_idx[
            self.labels[idx]
        ]

        return image, label


train_dataset = BrainTumorDataset(
    train_files,
    train_labels,
    transform=train_transform
)

val_dataset = BrainTumorDataset(
    val_files,
    val_labels,
    transform=val_test_transform
)

test_dataset = BrainTumorDataset(
    test_files,
    test_labels,
    transform=val_test_transform
)


BATCH_SIZE = 32


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


print(
    "Train dataset:",
    len(train_dataset)
)

print(
    "Validation dataset:",
    len(val_dataset)
)

print(
    "Test dataset:",
    len(test_dataset)
)

print(
    "Batch size:",
    BATCH_SIZE
)

Train dataset: 5040
Validation dataset: 1080
Test dataset: 1080
Batch size: 32


In [7]:
# LOAD MRI-TRAINED EFFICIENTNET-B0

efficientnet = models.efficientnet_b0(
    weights=None
)

FEATURE_DIM = (
    efficientnet.classifier[1]
    .in_features
)

print(
    "EfficientNet feature dimension:",
    FEATURE_DIM
)

assert FEATURE_DIM == 1280


# RECREATE ORIGINAL 4-CLASS CLASSIFIER
# SO THE SAVED CHECKPOINT CAN BE LOADED

efficientnet.classifier[1] = nn.Linear(
    FEATURE_DIM,
    4
)


efficientnet_state = torch.load(
    EFFICIENTNET_CHECKPOINT,
    map_location="cpu",
    weights_only=True
)


efficientnet.load_state_dict(
    efficientnet_state,
    strict=True
)


print(
    "MRI-trained EfficientNet-B0 loaded successfully."
)

EfficientNet feature dimension: 1280
MRI-trained EfficientNet-B0 loaded successfully.


In [8]:
# COMPRESSED CLASSICAL MODEL

class CompressedClassicalModel(nn.Module):

    def __init__(
        self,
        trained_efficientnet
    ):

        super().__init__()

        # SAME MRI-TRAINED FEATURE EXTRACTOR
        self.features = (
            trained_efficientnet.features
        )

        self.avgpool = (
            trained_efficientnet.avgpool
        )


        # FREEZE COMPLETE EFFICIENTNET BACKBONE
        # SAME AS HQNN

        for param in self.features.parameters():

            param.requires_grad = False


        # 1280 → 4
        self.feature_reduction = nn.Linear(
            1280,
            4
        )


        # 4 → 4 CLASSES
        self.classifier = nn.Linear(
            4,
            4
        )


    def forward(self, x):

        # EFFICIENTNET FEATURES

        x = self.features(x)

        x = self.avgpool(x)

        x = torch.flatten(
            x,
            1
        )


        # FEATURE COMPRESSION
        # 1280 → 4

        x = self.feature_reduction(
            x
        )


        # IMPORTANT:
        # NO tanh
        # NO multiplication by pi
        # NO quantum circuit


        # CLASSICAL CLASSIFICATION
        # 4 → 4

        x = self.classifier(
            x
        )

        return x


# RESET SEED BEFORE INITIALISING NEW HEAD

set_global_seed(SEED)


compressed_model = (
    CompressedClassicalModel(
        efficientnet
    )
)


print(
    compressed_model
)

CompressedClassicalModel(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2

In [9]:
# VERIFY MODEL CONFIGURATION

backbone_trainable = sum(
    p.numel()
    for p in compressed_model.features.parameters()
    if p.requires_grad
)


trainable_parameters = sum(
    p.numel()
    for p in compressed_model.parameters()
    if p.requires_grad
)


print(
    "Trainable EfficientNet parameters:",
    backbone_trainable
)

print(
    "Total trainable parameters:",
    trainable_parameters
)


print("\nTrainable layers:")

for name, parameter in compressed_model.named_parameters():

    if parameter.requires_grad:

        print(
            name,
            parameter.shape,
            parameter.numel()
        )


assert backbone_trainable == 0
assert trainable_parameters == 5144


print(
    "\nModel configuration is correct."
)

Trainable EfficientNet parameters: 0
Total trainable parameters: 5144

Trainable layers:
feature_reduction.weight torch.Size([4, 1280]) 5120
feature_reduction.bias torch.Size([4]) 4
classifier.weight torch.Size([4, 4]) 16
classifier.bias torch.Size([4]) 4

Model configuration is correct.


In [10]:
# TRAIN COMPRESSED CLASSICAL MODEL

from sklearn.metrics import f1_score
import time

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

compressed_model = compressed_model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    [
        *compressed_model.feature_reduction.parameters(),
        *compressed_model.classifier.parameters()
    ],
    lr=0.001
)

NUM_EPOCHS = 10

best_val_loss = float("inf")
best_epoch = 0

train_losses = []
train_accuracies = []
train_f1_scores = []

val_losses = []
val_accuracies = []
val_f1_scores = []

start_time = time.time()


for epoch in range(NUM_EPOCHS):

    set_global_seed(SEED + epoch)

    # --------------------
    # TRAINING
    # --------------------

    compressed_model.train()

    # Keep frozen EfficientNet in evaluation mode
    compressed_model.features.eval()

    running_loss = 0.0
    train_predictions = []
    train_targets = []

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = compressed_model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()
        optimizer.step()

        running_loss += (
            loss.item()
            * images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_predictions.extend(
            predictions.cpu().numpy()
        )

        train_targets.extend(
            labels.cpu().numpy()
        )


    train_loss = (
        running_loss
        / len(train_dataset)
    )

    train_accuracy = accuracy_score(
        train_targets,
        train_predictions
    )

    train_f1 = f1_score(
        train_targets,
        train_predictions,
        average="macro"
    )


    # --------------------
    # VALIDATION
    # --------------------

    compressed_model.eval()

    running_val_loss = 0.0
    val_predictions = []
    val_targets = []


    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = compressed_model(
                images
            )

            loss = criterion(
                outputs,
                labels
            )

            running_val_loss += (
                loss.item()
                * images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_predictions.extend(
                predictions.cpu().numpy()
            )

            val_targets.extend(
                labels.cpu().numpy()
            )


    val_loss = (
        running_val_loss
        / len(val_dataset)
    )

    val_accuracy = accuracy_score(
        val_targets,
        val_predictions
    )

    val_f1 = f1_score(
        val_targets,
        val_predictions,
        average="macro"
    )


    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)
    train_f1_scores.append(train_f1)

    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)
    val_f1_scores.append(val_f1)


    # SAVE BEST MODEL
    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch + 1

        torch.save(
            compressed_model.state_dict(),
            "/content/BEST_COMPRESSED_CLASSICAL.pth"
        )

        marker = " <-- BEST"

    else:
        marker = ""


    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Train F1: {train_f1:.4f}"
    )

    print(
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val F1: {val_f1:.4f}"
        f"{marker}"
    )


training_time = time.time() - start_time

print("\nTraining completed.")
print("Best epoch:", best_epoch)
print(
    "Best validation loss:",
    round(best_val_loss, 4)
)
print(
    "Training time:",
    round(training_time, 2),
    "seconds"
)

# DOWNLOAD THE BEST MODEL DIRECTLY TO YOUR COMPUTER

from google.colab import files

MODEL_PATH = "/content/BEST_COMPRESSED_CLASSICAL.pth"

print("\nBest model saved:")
print("Best epoch:", best_epoch)
print("Best validation loss:", round(best_val_loss, 4))

print("\nDownloading BEST_COMPRESSED_CLASSICAL.pth...")

files.download(MODEL_PATH)

Epoch 1/10 | Train Loss: 0.3625 | Train Acc: 0.9194 | Train F1: 0.9192
Val Loss: 0.1731 | Val Acc: 0.9491 | Val F1: 0.9487 <-- BEST
Epoch 2/10 | Train Loss: 0.1061 | Train Acc: 0.9740 | Train F1: 0.9740
Val Loss: 0.1173 | Val Acc: 0.9685 | Val F1: 0.9684 <-- BEST
Epoch 3/10 | Train Loss: 0.0715 | Train Acc: 0.9804 | Train F1: 0.9804
Val Loss: 0.1042 | Val Acc: 0.9676 | Val F1: 0.9675 <-- BEST
Epoch 4/10 | Train Loss: 0.0617 | Train Acc: 0.9825 | Train F1: 0.9825
Val Loss: 0.0932 | Val Acc: 0.9704 | Val F1: 0.9703 <-- BEST
Epoch 5/10 | Train Loss: 0.0497 | Train Acc: 0.9855 | Train F1: 0.9855
Val Loss: 0.0890 | Val Acc: 0.9750 | Val F1: 0.9749 <-- BEST
Epoch 6/10 | Train Loss: 0.0460 | Train Acc: 0.9859 | Train F1: 0.9859
Val Loss: 0.0869 | Val Acc: 0.9694 | Val F1: 0.9694 <-- BEST
Epoch 7/10 | Train Loss: 0.0399 | Train Acc: 0.9901 | Train F1: 0.9901
Val Loss: 0.0843 | Val Acc: 0.9704 | Val F1: 0.9703 <-- BEST
Epoch 8/10 | Train Loss: 0.0370 | Train Acc: 0.9887 | Train F1: 0.9887
Val L

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
# FINAL TESTING OF THE BEST COMPRESSED CLASSICAL MODEL

import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# DEVICE
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Evaluation device:", device)

# LOAD THE BEST SAVED MODEL
compressed_model.load_state_dict(
    torch.load(
        "/content/BEST_COMPRESSED_CLASSICAL.pth",
        map_location=device,
        weights_only=True
    )
)

compressed_model = compressed_model.to(device)
compressed_model.eval()

print("Best compressed classical checkpoint loaded successfully.")


# STORAGE
test_targets = []
test_predictions = []
test_probabilities = []

running_test_loss = 0.0


# TEST
with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = compressed_model(images)

        loss = criterion(
            outputs,
            labels
        )

        running_test_loss += (
            loss.item() * images.size(0)
        )

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        test_targets.extend(
            labels.cpu().numpy()
        )

        test_predictions.extend(
            predictions.cpu().numpy()
        )

        test_probabilities.extend(
            probabilities.cpu().numpy()
        )


# CONVERT TO NUMPY
test_targets = np.array(test_targets)
test_predictions = np.array(test_predictions)
test_probabilities = np.array(test_probabilities)


# METRICS
test_loss = (
    running_test_loss
    / len(test_dataset)
)

test_accuracy = accuracy_score(
    test_targets,
    test_predictions
)

test_precision = precision_score(
    test_targets,
    test_predictions,
    average="macro"
)

test_recall = recall_score(
    test_targets,
    test_predictions,
    average="macro"
)

test_f1 = f1_score(
    test_targets,
    test_predictions,
    average="macro"
)

test_roc_auc = roc_auc_score(
    test_targets,
    test_probabilities,
    multi_class="ovr",
    average="macro"
)


# RESULTS
print("\nFINAL COMPRESSED CLASSICAL TEST RESULTS")

print(
    f"Test Loss:       {test_loss:.4f}"
)

print(
    f"Accuracy:        {test_accuracy:.4f} "
    f"({test_accuracy * 100:.2f}%)"
)

print(
    f"Macro Precision: {test_precision:.4f} "
    f"({test_precision * 100:.2f}%)"
)

print(
    f"Macro Recall:    {test_recall:.4f} "
    f"({test_recall * 100:.2f}%)"
)

print(
    f"Macro F1:        {test_f1:.4f} "
    f"({test_f1 * 100:.2f}%)"
)

print(
    f"Macro ROC-AUC:   {test_roc_auc:.4f} "
    f"({test_roc_auc * 100:.2f}%)"
)


# CLASSIFICATION REPORT
print("\nCLASSIFICATION REPORT\n")

print(
    classification_report(
        test_targets,
        test_predictions,
        target_names=[
            "Glioma",
            "Meningioma",
            "No Tumor",
            "Pituitary"
        ],
        digits=4
    )
)


# CONFUSION MATRIX
cm = confusion_matrix(
    test_targets,
    test_predictions
)

print("\nCONFUSION MATRIX")
print(cm)

Evaluation device: cuda
Best compressed classical checkpoint loaded successfully.

FINAL COMPRESSED CLASSICAL TEST RESULTS
Test Loss:       0.1379
Accuracy:        0.9556 (95.56%)
Macro Precision: 0.9555 (95.55%)
Macro Recall:    0.9556 (95.56%)
Macro F1:        0.9554 (95.54%)
Macro ROC-AUC:   0.9962 (99.62%)

CLASSIFICATION REPORT

              precision    recall  f1-score   support

      Glioma     0.9579    0.9259    0.9416       270
  Meningioma     0.9254    0.9185    0.9219       270
    No Tumor     0.9607    0.9963    0.9782       270
   Pituitary     0.9779    0.9815    0.9797       270

    accuracy                         0.9556      1080
   macro avg     0.9555    0.9556    0.9554      1080
weighted avg     0.9555    0.9556    0.9554      1080


CONFUSION MATRIX
[[250  16   4   0]
 [ 10 248   6   6]
 [  1   0 269   0]
 [  0   4   1 265]]


In [12]:
class_to_idx = {
    "glioma": 0,
    "meningioma": 1,
    "notumor": 2,
    "pituitary": 3
}


train_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


val_test_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


class BrainTumorDataset(Dataset):

    def __init__(
        self,
        files,
        labels,
        transform=None
    ):

        self.files = files
        self.labels = labels
        self.transform = transform


    def __len__(self):

        return len(self.files)


    def __getitem__(self, idx):

        image = Image.open(
            self.files[idx]
        ).convert("RGB")

        if self.transform:

            image = self.transform(image)

        label = class_to_idx[
            self.labels[idx]
        ]

        return image, label


train_dataset = BrainTumorDataset(
    train_files,
    train_labels,
    transform=train_transform
)

val_dataset = BrainTumorDataset(
    val_files,
    val_labels,
    transform=val_test_transform
)

test_dataset = BrainTumorDataset(
    test_files,
    test_labels,
    transform=val_test_transform
)


BATCH_SIZE = 32


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))
print("Batch size:", BATCH_SIZE)

Train: 5040
Validation: 1080
Test: 1080
Batch size: 32
